In [1]:
%load_ext autoreload
%autoreload 2
import logging
import numpy as np
import pandas as pd
from pathlib import Path
from scentree.io.writer import save_json
from scentree.io.loader import Dataset, DatasetsLoader
from scentree.fan_generator import StageManager
from scentree.tree_construction.ftc import FTC

logging.basicConfig(level=logging.INFO)

SEED = 42
np.random.seed(SEED)  # global seed: FTC.generate_scenario_trees() uses np.random internally, no seed param

/users/delfos/aina/scentree-gen-remote/scentree-gen-remote/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Real data

### Loader

In [2]:
is_15 = True
if is_15:
    data_folder = Path("data_15min")
else:
    data_folder = Path("data_60min")
dam = pd.read_csv(data_folder / "DA.txt", sep="\t", header=None, index_col=0, parse_dates=[0])
rm = pd.read_csv(data_folder / "RM.txt", sep="\t", header=None, index_col=0, parse_dates=[0])
im1 = pd.read_csv(data_folder / "IM1.txt", sep="\t", header=None, index_col=0, parse_dates=[0])
im2 = pd.read_csv(data_folder / "IM2.txt", sep="\t", header=None, index_col=0, parse_dates=[0])
wind = pd.read_csv(data_folder / "WP.txt", sep="\t", header=None, index_col=0, parse_dates=[0])
solar = pd.read_csv(data_folder / "PV.txt", sep="\t", header=None, index_col=0, parse_dates=[0])
im3 = pd.read_csv(data_folder / "IM3.txt", sep="\t", header=None, index_col=0, parse_dates=[0])
ib_up = pd.read_csv(data_folder / "IB_UP.txt", sep="\t", header=None, index_col=0, parse_dates=[0])
ib_down = pd.read_csv(data_folder / "IB_DOWN.txt", sep="\t", header=None, index_col=0, parse_dates=[0])
print(dam.shape, rm.shape, im1.shape, im2.shape, wind.shape, solar.shape, im3.shape, ib_up.shape, ib_down.shape)

(577, 96) (577, 96) (577, 96) (577, 96) (577, 96) (577, 96) (485, 48) (577, 96) (577, 96)


In [3]:

#Filter dates
dam = dam[dam.index < "2025-12-01"]
rm = rm[rm.index < "2025-12-01"]
im1 = im1[im1.index < "2025-12-01"]
wind = wind[wind.index < "2025-12-01"]
im2 = im2[im2.index < "2025-12-01"]
solar = solar[solar.index < "2025-12-01"]
im3 = im3[im3.index < "2025-12-01"]
ib_up = ib_up[ib_up.index < "2025-12-01"]
ib_down = ib_down[ib_down.index < "2025-12-01"]


In [4]:
if is_15:
    renewable_stages = [i for i in range(5, 102) if i != 45 for _ in range(1)]
else:
    renewable_stages = [i for i in range(5, 30) if i != 15 for _ in range(1)]
print(renewable_stages)

[5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101]


In [5]:
datasets = [
    Dataset(
        name="DA",
        values=dam.values,
        stage_ids=[1] * dam.shape[1],
    ),
    Dataset(
        name="RM",
        values=rm.values,
        stage_ids=[2] * rm.shape[1],
    ),
    Dataset(
        name="IM1",
        values=im1.values,
        stage_ids=[3] * im1.shape[1],
    ),
    Dataset(
        name="IM2",
        values=im2.values,
        stage_ids=[4] * im2.shape[1],
    ),
    Dataset(
        name="WP",
        values=wind.values,
        stage_ids=renewable_stages,
        bounds=(0,1),
    ),
    Dataset(
        name="PV",
        values=solar.values,
        stage_ids=renewable_stages,
        bounds=(0,1),
    ),
    Dataset(
        name="IM3",
        values=im3.values,
        stage_ids=[45] * im3.shape[1],
    ),
    Dataset(
        name="IB_UP",
        values=ib_up.values,
        stage_ids=[102] * ib_up.shape[1],
    ),
    Dataset(
        name="IB_DOWN",
        values=ib_down.values,
        stage_ids=[102] * ib_down.shape[1],
    ),
]
dataset_loader = DatasetsLoader(datasets=datasets)
full_bounds = dataset_loader.get_full_bounds()
full_values = dataset_loader.get_full_values()
num_variables_per_stage = dataset_loader.get_num_variables_per_stage()
stage_ids = dataset_loader.get_sorted_stage_ids()
map_columns_names = dataset_loader.create_stages_columns_mapping()

### Scenario fan

In [6]:
# Scenario fan
num_fans = 31
build_in_sample_fans = True
stage_manager = StageManager()
for num_scenarios in range(50,301,10):
    scenario_fans = stage_manager.generate_scenario_fans(
        X=full_values,
        num_fans=num_fans,
        num_scenarios=num_scenarios,
        build_in_sample_fans=build_in_sample_fans,
        value_ranges=full_bounds,
        seed=SEED,
    )
    tree_builder = FTC(
        scenarios=scenario_fans["scenarios"],
        num_variables_per_stage=num_variables_per_stage,
        stage_ids=stage_ids
    )
    scenario_trees = tree_builder.generate_scenario_trees(r=2, initial_stage_id_to_cluster=1)
    save_json(
        output_dir="./dif_ren_scentree",
        num_stages=len(stage_ids),
        in_sample_prediction=build_in_sample_fans,
        predicted_value=scenario_fans["predicted_values"],
        observed_value=scenario_fans["observed_values"],
        scenario_trees=scenario_trees,
        mapping_datasets_columns=map_columns_names,
        multiple_files=True,
        name = f"scenariotree_{num_scenarios}"
    )

Evaluating estimators:   0%|          | 0/2 [00:00<?, ?it/s]

Evaluating estimators:   0%|          | 0/2 [00:00<?, ?it/s, estimator=RidgeEstimator]

Evaluating estimators:  50%|█████     | 1/2 [00:02<00:02,  2.23s/it, estimator=RidgeEstimator]

Evaluating estimators:  50%|█████     | 1/2 [00:02<00:02,  2.23s/it, estimator=VarEstimator]  

Evaluating estimators: 100%|██████████| 2/2 [00:02<00:00,  1.02s/it, estimator=VarEstimator]

...

Building trees: 100%|██████████| 31/31 [05:03<00:00,  9.77s/it]

Writing files:   0%|          | 0/31 [00:00<?, ?it/s]

Writing files:  58%|█████▊    | 18/31 [00:00<00:00, 167.00it/s]

Writing files: 100%|██████████| 31/31 [00:00<00:00, 157.32it/s]

INFO:scentree.io.writer:Results saved in dif_ren_scentree/scenariotree_300


In [7]:
#scenario_fans["scenarios"][0].shape

### Scenario Tree

In [8]:
"""
tree_builder = FTC(
    scenarios=scenario_fans["scenarios"],
    num_variables_per_stage=num_variables_per_stage,
    stage_ids=stage_ids
)
scenario_trees = tree_builder.generate_scenario_trees(r=2, initial_stage_id_to_cluster=1)
"""

'\ntree_builder = FTC(\n    scenarios=scenario_fans["scenarios"],\n    num_variables_per_stage=num_variables_per_stage,\n    stage_ids=stage_ids\n)\nscenario_trees = tree_builder.generate_scenario_trees(r=2, initial_stage_id_to_cluster=1)\n'

### Output

In [9]:
"""
save_json(
    output_dir="./dif_ren_scentree",
    num_stages=len(stage_ids),
    in_sample_prediction=build_in_sample_fans,
    predicted_value=scenario_fans["predicted_values"],
    observed_value=scenario_fans["observed_values"],
    scenario_trees=scenario_trees,
    mapping_datasets_columns=map_columns_names,
    multiple_files=True,
    name = f"scenariotree_{num_scenarios}"
)
"""

'\nsave_json(\n    output_dir="./dif_ren_scentree",\n    num_stages=len(stage_ids),\n    in_sample_prediction=build_in_sample_fans,\n    predicted_value=scenario_fans["predicted_values"],\n    observed_value=scenario_fans["observed_values"],\n    scenario_trees=scenario_trees,\n    mapping_datasets_columns=map_columns_names,\n    multiple_files=True,\n    name = f"scenariotree_{num_scenarios}"\n)\n'